# Paper 04 · AlexNet Design Claims

**Citation:** Alex Krizhevsky, Ilya Sutskever, Geoffrey Hinton, “ImageNet Classification with Deep Convolutional Neural Networks” (NeurIPS 2012).

**Paper:** https://papers.nips.cc/paper/4824-imagenet-classification-with-deep-convolutional-neural-networks.pdf

> **Scale gap:** We do not reproduce ImageNet. We test three design choices on the 8×8 digits dataset: ReLU vs tanh, augmentation, and dropout.

## Mathematical Framework

Before reproducing the paper experimentally, work through the relevant mathematical companions:

- [Math 01 · Linear Algebra & Geometry](../../math/01_linear_algebra_geometry.ipynb)
- [Math 02 · Calculus & Matrix Calculus](../../math/02_calculus_matrix_calculus.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 08 · Regularization & Generalization](../../math/08_regularization_generalization.ipynb)
- [Math 10 · Neural-Network Mathematics](../../math/10_neural_network_math.ipynb)

For the paper defense, be able to explain the **objective, derivation, assumptions, and why the reported mechanism should follow mathematically**, not just what the code did.

## Before you read
1. Which contributions are architectural versus optimization/regularization choices?
2. Why might ReLU train faster than saturating nonlinearities?
3. How do augmentation and dropout attack different failure modes?

## Central claim
A deep CNN trained efficiently on GPUs, using ReLU, data augmentation, and dropout, achieved a major ImageNet accuracy improvement.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
from coursekit.experiments import Experiment
experiment = Experiment('paper-04_alexnet', {'bootstrap_seed': SEED, 'scope': 'educational mechanism demonstration', 'note': 'Original notebook may use additional explicit seeds; source hash records the exact experiment.'}, source='papers/notebooks/04_alexnet.ipynb')
experiment.capture_figures()

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
d=load_digits()
X=torch.tensor((d.images/16).astype("float32")[:,None]); y=torch.tensor(d.target.astype("int64"))
idx=np.arange(len(y)); tr,te=train_test_split(idx,test_size=.3,random_state=1,stratify=y.numpy())
Xtr,Xte,ytr,yte=X[tr],X[te],y[tr],y[te]

## Build variants
This is a **design-claim reproduction**, not AlexNet itself.

In [ ]:
class Net(nn.Module):
    def __init__(self,activation="relu",dropout=0.0):
        super().__init__()
        A=nn.ReLU if activation=="relu" else nn.Tanh
        self.net=nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),A(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),A(),nn.Flatten(),
            nn.Linear(32*4*4,64),A(),nn.Dropout(dropout),nn.Linear(64,10))
    def forward(self,x): return self.net(x)

def jitter(x):
    # TODO: try additional label-preserving augmentation.
    shift=int(torch.randint(-1,2,(1,)))
    return torch.roll(x,shifts=shift,dims=3)

## Run ablations

In [ ]:
def run(act="relu",dropout=0,augment=False,epochs=30):
    torch.manual_seed(0); m=Net(act,dropout); opt=torch.optim.Adam(m.parameters(),lr=.005); ce=nn.CrossEntropyLoss(); hist=[]
    for _ in range(epochs):
        m.train(); xb=jitter(Xtr) if augment else Xtr
        opt.zero_grad(); loss=ce(m(xb),ytr); loss.backward(); opt.step()
        m.eval()
        with torch.no_grad(): acc=(m(Xte).argmax(1)==yte).float().mean().item()
        hist.append([loss.item(),acc])
    return np.array(hist)

variants={
 "tanh":run("tanh",0,False),
 "relu":run("relu",0,False),
 "relu+dropout":run("relu",.4,False),
 "relu+augmentation":run("relu",0,True)
}
for name,h in variants.items():
    print(name,"final acc",h[-1,1])
    plt.plot(h[:,0],label=name)
plt.yscale("log"); plt.legend(); plt.title("Training loss by design choice"); plt.show()

### Reading challenge
Which of these results are dataset-specific? Why would success on tiny digits not establish the original ImageNet claim?

## Ablation table
Fill this after running the experiments.

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
Answer without looking back at the notebook:
1. What problem existed before this work?
2. What was actually new?
3. What evidence in your reproduction supports the central claim?
4. What does your reduced-scale reproduction **not** establish?
5. Which idea from this paper survived into modern systems?
6. What experiment would you run next?

## Evidence export

Figures and numeric diagnostics are captured. Explicit metrics use `experiment.log(variant, seed, metrics)`. Use `run_trials` for paired-seed ablations. An empty metrics table or `not_run` ablation is incomplete evidence, not success. Interpretations remain your work.

In [ ]:
print('Evidence directory:', experiment.finish(globals()))